In [ ]:
# pip install -U git+https://github.com/cvai-roig-lab/Net2Brain

In [1]:
import net2brain
import cv2
import os, glob, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import decimate
from net2brain.feature_extraction import FeatureExtractor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
import joblib
from pathlib import Path

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=LinAlgWarning)

/Users/yetkin/Desktop/Hackathon/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [4]:
BASE_DIR     = Path('../eeg-mind-reading-challenge')
STIM_DIR     = BASE_DIR / 'stimuli_train'  
FEAT_DIR     = BASE_DIR / 'features'       
TEMP_FEAT    = BASE_DIR / 'temp_features' 
TRAIN_DIR    = BASE_DIR / 'trn'
ARTIFACT_DIR = Path('../artifacts')

SFREQ = 500
DEC = 5
N_FRAMES = 8

def features_for(ids, layer_path):
    feat_path = FEAT_DIR / f'video_{layer_path}.npz'
    z = np.load(feat_path)
    rows = []
    for sid in ids:
        key = f'{int(sid):05d}'
        assert key in z.files, f'No features found for stimulus {sid}'
        rows.append(np.array(z[key], dtype=np.float32))   
    return np.stack(rows)

vids = sorted(list(STIM_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_DIR}")
all_stim_ids = np.arange(1, len(vids) + 1)

print("Loading all unique video features for global PCA...")
X_unique_alex = features_for(all_stim_ids, 'features.3')
X_unique_dino = features_for(all_stim_ids, 'blocks.6')
X_unique_clip = features_for(all_stim_ids, 'visual.transformer.resblocks.7')

print("Preprocessing unique video features...")
sc = StandardScaler().fit(X_unique_alex)
X_trn_alex_z = sc.transform(X_unique_alex)
sc2 = StandardScaler().fit(X_unique_dino)
X_trn_dino_z = sc2.transform(X_unique_dino)
sc3 = StandardScaler().fit(X_unique_clip)
X_trn_clip_z = sc3.transform(X_unique_clip)

X_trn_ensemble = np.hstack([X_trn_alex_z, X_trn_dino_z, X_trn_clip_z])

pca = PCA(n_components=0.95, svd_solver='full', random_state=42).fit(X_trn_ensemble)
X_unique_p = pca.transform(X_trn_ensemble)

sc4 = StandardScaler().fit(X_unique_p)
X_unique_s = sc4.transform(X_unique_p)

# Build the required lookup dictionary
video_feature_map = {int(sid): X_unique_s[i] for i, sid in enumerate(all_stim_ids)}
print("Video feature map successfully created!")

joblib.dump({
    'sc': sc, 'pca': pca, 'sc2': sc2, 'sc3': sc3, 'sc4': sc4}, 
    ARTIFACT_DIR / f'ridge_regression_sub-global.joblib', compress=3)

alphas = np.logspace(-3, 6, 10)
# --- 2. Loop Through Each Subject Individually ---
for sub in range(1, 7):
    trn_path = TRAIN_DIR / f'sub-{sub:02d}_trn.npz'
    if not os.path.exists(trn_path):
        print(f"Skipping Subject {sub:02d} (not found)")
        continue
        
    print(f"\n==============================")
    print(f"Training Model for Subject {sub:02d}")
    print(f"==============================")
    
    trn = np.load(trn_path)
    trn_ids = np.array(sorted([int(k) for k in trn.files], key=int), dtype=int)
    trn_eeg = np.stack([trn[str(i)] for i in trn_ids])
    
    trn_eeg = decimate(trn_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
    n_stim, n_chan, n_time = trn_eeg.shape

    X_trn_s = np.array([video_feature_map[int(sid)] for sid in trn_ids])

    model = RidgeCV(alphas=alphas, cv=5)
    model.fit(X_trn_s, trn_eeg.reshape(len(trn_ids), -1))
    print(f"Subject {sub:02d} Best alpha: {model.alpha_}")

    # --- 4. Evaluate Subject Performance ---
    pred_eeg = model.predict(X_trn_s)
    Y_pred_3d = pred_eeg.reshape(len(trn_ids), n_chan, n_time)

    M = trn_eeg[:, :, 5:60].reshape(len(trn_ids), -1)
    P = Y_pred_3d[:, :, 5:60].reshape(len(trn_ids), -1)
    Mz = (M - M.mean(1, keepdims=True)) / M.std(1, keepdims=True)
    Pz = (P - P.mean(1, keepdims=True)) / P.std(1, keepdims=True)

    S = Mz @ Pz.T / M.shape[1]
    t = np.arange(len(trn_ids))
    ranks = (-S).argsort(1).argsort(1)[t, t] + 1
    print(f"Subject {sub:02d} -> Top-1: {(ranks == 1).mean():.4f} | Top-5: {(ranks <= 5).mean():.4f}")

    # --- 5. Save Subject-Specific Artifacts ---
    joblib.dump({
        'model': model,
        'n_chan': n_chan, 'n_time': n_time,
        'DEC': DEC, 'SFREQ': SFREQ, 'N_FRAMES': N_FRAMES}, 
        ARTIFACT_DIR / f'ridge_regression_sub-{sub:02d}.joblib', compress=3)

Found 1000 total videos in ../eeg-mind-reading-challenge/stimuli_train
Loading all unique video features for global PCA...
Preprocessing unique video features...
Video feature map successfully created!

Training Model for Subject 01
Subject 01 Best alpha: 10000.0
Subject 01 -> Top-1: 0.8977 | Top-5: 0.9298

Training Model for Subject 02
Subject 02 Best alpha: 100000.0
Subject 02 -> Top-1: 0.7253 | Top-5: 0.8282

Training Model for Subject 03
Subject 03 Best alpha: 10000.0
Subject 03 -> Top-1: 0.9063 | Top-5: 0.9365

Training Model for Subject 04
Subject 04 Best alpha: 100000.0
Subject 04 -> Top-1: 0.7476 | Top-5: 0.8448

Training Model for Subject 05
Subject 05 Best alpha: 10000.0
Subject 05 -> Top-1: 0.8929 | Top-5: 0.9254

Training Model for Subject 06
Subject 06 Best alpha: 100000.0
Subject 06 -> Top-1: 0.8059 | Top-5: 0.8659


# TESTING

## Single

In [7]:
BASE_DIR         = Path('../eeg-mind-reading-challenge')
TEST_DIR         = BASE_DIR / 'tst'
STIM_TEST_DIR    = BASE_DIR / 'stimuli_test'
FRAME_TEST_DIR   = BASE_DIR / 'frames_test'
FEAT_TEST_DIR    = BASE_DIR / 'features_test'
TEMP_FEAT        = BASE_DIR / 'temp_features'
TEST_STIM        = BASE_DIR / 'tst_stim'
ARTIFACT_DIR     = Path('../artifacts')
PREDICTIONS_DIR  = Path('../predictions')

def features_for(ids, layer_path):
    feat_path = FEAT_TEST_DIR / f'video_{layer_path}.npz'
    z = np.load(feat_path)
    rows = []
    for sid in ids:
        key = f'{int(sid):05d}'
        assert key in z.files, f'No features found for stimulus {sid}'
        rows.append(np.array(z[key], dtype=np.float32))   
    return np.stack(rows)

global_artifact_path = ARTIFACT_DIR / 'ridge_regression_sub-global.joblib'
B = joblib.load(global_artifact_path)
sc, sc2, sc3, sc4, pca = B['sc'], B['sc2'], B['sc3'], B['sc4'], B['pca']

vids = sorted(list(STIM_TEST_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_TEST_DIR}")
all_stim_ids = np.arange(1001, 1001 + len(vids))

print("Loading all unique video features for global PCA...")
X_unique_alex = features_for(all_stim_ids, 'features.3')
X_unique_dino = features_for(all_stim_ids, 'blocks.6')
X_unique_clip = features_for(all_stim_ids, 'visual.transformer.resblocks.7')

print("Preprocessing unique video features...")
X_trn_alex_z = sc.transform(X_unique_alex)
X_trn_dino_z = sc2.transform(X_unique_dino)
X_trn_clip_z = sc3.transform(X_unique_clip)
X_trn_ensemble = np.hstack([X_trn_alex_z, X_trn_dino_z, X_trn_clip_z])
X_unique_p = pca.transform(X_trn_ensemble)
X_unique_s = sc4.transform(X_unique_p)

# Build the required lookup dictionary
video_feature_map = {int(sid): X_unique_s[i] for i, sid in enumerate(all_stim_ids)}
print("Video feature map successfully created!")

for sub in range(1, 7):
    tst_path = TEST_DIR / f'sub-{sub:02d}_tst.npy'
    model_path = ARTIFACT_DIR / f'ridge_regression_sub-{sub:02d}.joblib'
    if not tst_path.exists() or not model_path.exists():
        print(f"\nSkipping Subject {sub:02d} (file or model not found)")
        continue

    print(f"\n==============================")
    print(f"Testing Subject {sub:02d}")
    print(f"==============================")

    B = joblib.load(model_path)
    model = B['model']
    n_chan, n_time = B['n_chan'], B['n_time']
    DEC = B['DEC']

    tst_ids = list(all_stim_ids)
    tst_eeg = np.load(tst_path)
    tst_eeg = decimate(tst_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
    n_test_trials = len(tst_ids)
    
    X_tst_s = np.array([video_feature_map[int(sid)] for sid in tst_ids])

    pred_tst_eeg = model.predict(X_tst_s)
    reshaped_pred_tst_eeg = pred_tst_eeg.reshape(n_test_trials, n_chan, n_time)
    print(reshaped_pred_tst_eeg.shape)

    gt_path = TEST_STIM / f'sub-{sub:02d}_tst_stim.npy'
    has_ground_truth = os.path.exists(gt_path)

    if has_ground_truth:
        gt = np.load(gt_path)
        order = np.argsort(gt) 
        M = tst_eeg[order][:, :, 5:60].reshape(n_test_trials, -1)               
    else:
        M = tst_eeg[:, :, 5:60].reshape(n_test_trials, -1)    
        print(f"Ground truth not found for Subject {sub:02d}. Generating blind candidate predictions...")

    P = reshaped_pred_tst_eeg[:, :, 5:60].reshape(102, -1)
    Mz = (M - M.mean(1, keepdims=True)) / M.std(1, keepdims=True)
    Pz = (P - P.mean(1, keepdims=True)) / P.std(1, keepdims=True)

    S = Mz @ Pz.T / M.shape[1]                         

    video_ids = np.array(tst_ids)              # column j -> video 1001+j
    order_idx = np.argsort(-S, axis=1)          # (102, 102) column indices, best first
    ranked = video_ids[order_idx].astype(np.int32)          # (102, 102) video IDs, best first

    np.save(PREDICTIONS_DIR / f'sub-0{sub}_ranked_candidate_list.npy', ranked)
    print(f"Saved ranked candidate list for Subject {sub:02d}")
    
    if has_ground_truth:
        t = np.arange(n_test_trials)
        ranks = (-S).argsort(1).argsort(1)[t, t] + 1
        print('top-1:', (ranks == 1).mean(), 'top-5:', (ranks <= 5).mean(), 'median:', np.median(ranks))

        truth_ids = gt[order]                       
        top1 = (ranked[:, 0] == truth_ids).mean()
        top5 = (ranked[:, :5] == truth_ids[:, None]).any(axis=1).mean()

        print(f'Evaluated top-1: {top1:.4f}  (chance {1/n_test_trials:.4f})')
        print(f'Evaluated top-5: {top5:.4f}  (chance {5/n_test_trials:.4f})')

Found 102 total videos in ../eeg-mind-reading-challenge/stimuli_test
Loading all unique video features for global PCA...
Preprocessing unique video features...
Video feature map successfully created!

Testing Subject 01
(102, 17, 300)
Saved ranked candidate list for Subject 01
top-1: 0.058823529411764705 top-5: 0.24509803921568626 median: 15.0
Evaluated top-1: 0.0588  (chance 0.0098)
Evaluated top-5: 0.2451  (chance 0.0490)

Testing Subject 02
(102, 17, 300)
Saved ranked candidate list for Subject 02
top-1: 0.0392156862745098 top-5: 0.24509803921568626 median: 19.5
Evaluated top-1: 0.0392  (chance 0.0098)
Evaluated top-5: 0.2451  (chance 0.0490)

Testing Subject 03
(102, 17, 300)
Saved ranked candidate list for Subject 03
top-1: 0.049019607843137254 top-5: 0.17647058823529413 median: 20.5
Evaluated top-1: 0.0490  (chance 0.0098)
Evaluated top-5: 0.1765  (chance 0.0490)

Testing Subject 04
(102, 17, 300)
Saved ranked candidate list for Subject 04
top-1: 0.049019607843137254 top-5: 0.147

# RESULTS for subject 1

In [ ]:
# TRAIN:

# ALEX: top-1 0.78099173553719 top-5 0.8212809917355371 median 1.0
# DINO: top-1 0.9297520661157025 top-5 0.9566115702479339 median 1.0
# OPENCLIP, resblock6: top-1 0.9421487603305785 top-5 0.9597107438016529 median 1.0
# OPENCLIP, resblock7: top-1 0.8997933884297521 top-5 0.9297520661157025 median 1.0
# ensamble: top-1 0.9256198347107438 top-5 0.9514462809917356 median 1.0

# TEST:

# ALEX, features.3:  top-1 0.0392156862745098 top-5 0.11764705882352941 median 25.5
# DINO, blocks.6:      top-1 0.049019607843137254 top-5 0.30392156862745096 median 13.0
# CLIP, visual.transformer.resblocks.6:   top-1 0.0392156862745098 top-5 0.16666666666666666 median 17.0, (968, 38400)
# CLIP, visual.transformer.resblocks.7:   top-1 0.049019607843137254 top-5 0.21568627450980393 median 14.0 (968, 38400)
# Ensemble:  top-1 0.058823529411764705 top-5 0.2647058823529412 median 14.0
